In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tensorflow.keras.layers import LSTM, Dense, Masking, Input, Embedding, Concatenate, LayerNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
import keras
keras.utils.set_random_seed(28)
import datetime
from icecream import ic
from itertools import product
from sklearn.preprocessing import MinMaxScaler
import wandb
from configparser import ConfigParser
from sklearn.model_selection import train_test_split
from typing import Iterable

I0000 00:00:1777219411.595387  314117 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777219411.596183  314117 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777219411.651882  314117 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777219413.126232  314117 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [2]:
CUSTOM_DATASET_PATH = Path("custom_data/")

In [3]:
merged = pd.read_csv(CUSTOM_DATASET_PATH / "merged_2.csv")
merged

,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
2,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,3,8,2,2,5,0,18
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
511135,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


As the average stars review is 3.86, we will base our recommender only on positive reviews (at least 4 stars), as it is not our goal to only predict the next restaurants but also the next "good" restaurant. 

In [4]:
merged = merged[merged["stars_review"] >= 4]

We are filtering out users who only have one review as we need at least a 2 restaurant long sequence for our x and y split.

In [5]:
user_ids_to_filter_out = merged.value_counts("user_id")[merged.value_counts("user_id") < 2].index
user_ids_to_filter_out

Index(['6_SpY41LIHZuIaiDs5FMKA', 'WqfKtI-aGMmvbA9pPUxNQQ',
       'uk6po-UUCTk_NvKKgvsOwg', '0q2W3-ieBUJWD5TTLKi3Ug',
       'YqqSMPzBrZIng-Y0YJTvfw', 'nmW4jna8vbE50F9SgjmgPQ',
       '2Gp0gQNpIVmShIt3-gOebw', 'qVVPhYDSHsEfXQzklxfRKw',
       'u73tXwSsYPF04WiHP6pTng', 'pocYAxpIEGSCQxd37gNQ0w',
       ...
       'xm0GZBJjxcTivjIDFcBU3g', 'HluTYKUa2Q7nvSN7lxkgjw',
       '5_udBidKOG0Stho1nx0d6Q', 'ovcZpdYD2boCnqJlJX4gCg',
       'RNtWTnq4NT_vW_XVVXQxeA', 'ogGTGzUzKn1x5-ddTU4yfQ',
       'YqiMF5KzIw8xP494HNuKdw', '4rNNfQm1tB1xOVQ9ZduzNA',
       '7Ka3Smzb7hlHi8gk0wByjw', 'B7TD5yTemGv50y4wM2EVNA'],
      dtype='str', name='user_id', length=86467)

In [6]:
merged = merged[~ merged["user_id"].isin(user_ids_to_filter_out)]
merged.reset_index(inplace=True)
merged

/tmp/ipykernel_314117/889533434.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged.reset_index(inplace=True)


,index,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564,4.0,80,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,WnT9NIzQgLlILjPT0kEcsQ,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [7]:
# encoding business_id string to an integer
merged["business_id"], _ = pd.factorize(merged["business_id"])
# zero will be used as padding
merged["business_id"] += 1

In [8]:
merged

,index,business_id,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,1,39.955505,-75.155564,4.0,80,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,3432,39.935982,-75.158665,4.5,35,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [9]:
# encoding user_id string to an integer
merged["user_id"], _ = pd.factorize(merged["user_id"])

In [10]:
# merged.info(max_cols=500)

## MinMax Scaling
We will divide the dataframe into parts that are to be minmaxed (categories, attributes) and the rest (ids).

In [11]:
merged_no_ids = merged.drop(labels=["review_id", "user_id", "business_id"], axis=1)
display(merged_no_ids)

,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,1,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,6,1,0,11,28,46,46,26,20,186
1,3,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
2,4,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,0,1,4,24,51,35,35,13,5,141
3,5,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,4,0,0,41,34,51,51,26,97,669
4,7,39.955505,-75.155564,4.0,80,0,0,0,0,0,...,5,2,0,45,114,129,129,63,81,834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,511132,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,3
272829,511133,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
272830,511134,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,3,3,4,4,1,3,6
272831,511136,39.935982,-75.158665,4.5,35,0,0,0,0,0,...,0,0,0,5,0,0,0,0,0,77


In [12]:
merged_only_ids = merged[["review_id", "user_id", "business_id"]]
merged_only_ids

,review_id,user_id,business_id
0,uduvUCvi9w3T2bSGivCfXg,0,1
1,MKNp_CdR2k2202-c8GN5Dw,1,1
2,D1GisLDPe84Rrk_R4X2brQ,2,1
3,_hJu0u6nB-8LIeQJY4Vg4w,3,1
4,_xRGsS6QGpcD9LQJNtavuw,4,1
...,...,...,...
272828,cOh-a-xWgOBP4WHxVp2SOg,34500,3432
272829,qBcwQEQPnLxjkw-xbUIF4Q,33572,3432
272830,G8fbysnUAUmqq1XWTjMQ4Q,17780,3432
272831,JITY01bGbdsiUBznLz9rdg,7973,3432


In [13]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(merged_no_ids)
merged_no_ids = pd.DataFrame(scaler.transform(merged_no_ids), columns=merged_no_ids.columns, index=merged_no_ids.index)
display(merged_no_ids.head())

,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,Teppanyaki_category,Indian_category,Brasseries_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,0.000000,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001908,0.000783,0.000000,0.001863,0.001154,0.001959,0.001959,0.003021,0.000709,0.012338
1,0.000004,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
2,0.000006,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000783,0.003433,0.004064,0.002102,0.001491,0.001491,0.001510,0.000177,0.009337
3,0.000008,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001272,0.000000,0.000000,0.006942,0.001401,0.002172,0.002172,0.003021,0.003439,0.044551
4,0.000012,0.317734,0.512952,0.75,0.013121,0.0,0.0,0.0,0.0,0.0,...,0.001590,0.001566,0.000000,0.007619,0.004699,0.005495,0.005495,0.007320,0.002872,0.055556


In [14]:
merged = merged_only_ids.join(merged_no_ids)
display(merged)

,review_id,user_id,business_id,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
0,uduvUCvi9w3T2bSGivCfXg,0,1,0.000000,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001908,0.000783,0.000000,0.001863,0.001154,0.001959,0.001959,0.003021,0.000709,0.012338
1,MKNp_CdR2k2202-c8GN5Dw,1,1,0.000004,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000200
2,D1GisLDPe84Rrk_R4X2brQ,2,1,0.000006,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.000000,0.000783,0.003433,0.004064,0.002102,0.001491,0.001491,0.001510,0.000177,0.009337
3,_hJu0u6nB-8LIeQJY4Vg4w,3,1,0.000008,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001272,0.000000,0.000000,0.006942,0.001401,0.002172,0.002172,0.003021,0.003439,0.044551
4,_xRGsS6QGpcD9LQJNtavuw,4,1,0.000012,0.317734,0.512952,0.750,0.013121,0.0,0.0,...,0.001590,0.001566,0.000000,0.007619,0.004699,0.005495,0.005495,0.007320,0.002872,0.055556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272828,cOh-a-xWgOBP4WHxVp2SOg,34500,3432,0.999990,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000035,0.000133
272829,qBcwQEQPnLxjkw-xbUIF4Q,33572,3432,0.999992,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000041,0.000000,0.000000,0.000000,0.000000,0.000000
272830,G8fbysnUAUmqq1XWTjMQ4Q,17780,3432,0.999994,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000508,0.000124,0.000170,0.000170,0.000116,0.000106,0.000333
272831,JITY01bGbdsiUBznLz9rdg,7973,3432,0.999998,0.242793,0.505308,0.875,0.005248,0.0,0.0,...,0.000000,0.000000,0.000000,0.000847,0.000000,0.000000,0.000000,0.000000,0.000000,0.005069


## Splitting into sequences

In [15]:
sequence_length = 5

### Less than sequence length

In [16]:
def split_to_sequences_padding(
        df: pd.DataFrame,
        user_indices: Iterable,
        sequence_length: int,
        masking_value: int = -1
        ) -> list[pd.DataFrame]:
    """Split dataframe to sequences.

    Splits dataframe to time-series sequences based on user_id. Applies padding
    with masking_value to achieve the desired sequence_length. Business id is
    to be masked with zero (because of keras).

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be split into sequences.
    user_indices : Iterable
        Indices to be used what users are to be split into sequences.
    sequence_length : int
        The desired sequence length.
    masking_value : int, optional 
        The value to be used in padded restaurant reviews. By default -1.

    Returns
    -------
    list[pd.DataFrame]
        Return a list of sequences in form of a dataframe.
    """
    user_sequences = []
    for user_id in user_indices:
        user_df = df[df["user_id"] == user_id]
        # the first rows containing the "question"
        first_rows = user_df.head(n=len(user_df)-1)

        # the row containing the answer (predicted restaurant)
        last_row = user_df.tail(n=1)
        to_pad_length = sequence_length - len(user_df)
        to_pad = user_df.head(n=1)
        to_pad = pd.concat(to_pad_length * [to_pad])
        float_columns = to_pad.select_dtypes(include="float64").columns
        # zero is used for padding of restaurant ids
        to_pad["business_id"] = 0
        to_pad.loc[:, float_columns] = masking_value

        user_sequence = pd.concat([first_rows, to_pad, last_row])
        user_sequences.append(user_sequence)
    return user_sequences

In [17]:
indexes_less_than_sequence_length = merged.value_counts("user_id")[(merged.value_counts("user_id") > 1) & (merged.value_counts("user_id") < sequence_length)].index

In [18]:
user_sequences_padding = split_to_sequences_padding(df=merged, user_indices=indexes_less_than_sequence_length, sequence_length=sequence_length)

In [19]:
print("len(user_sequences_padding):", len(user_sequences_padding))

len(user_sequences_padding): 36176


In [20]:
user_matrix_padding = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in user_sequences_padding]
)

In [21]:
np.save(CUSTOM_DATASET_PATH / "user_matrix_padding_5", user_matrix_padding, allow_pickle=False)

In [22]:
user_matrix_padding = np.load(CUSTOM_DATASET_PATH / "user_matrix_padding_5.npy")

In [23]:
print("user_matrix_padding.shape:", user_matrix_padding.shape)

user_matrix_padding.shape: (36176, 5, 385)


### More or equal to sequence length

In [24]:
def split_to_sequences(
        df: pd.DataFrame,
        user_indices: Iterable,
        sequence_length: int
        ) -> tuple[list[pd.DataFrame], pd.DataFrame]:
    """
    Split dataframe into sequences.

    Splits dataframe to time-series sequences based on user_id. Assumed that all
    users have the sequence length of at least the sequence_length. Sequences
    that go over the limit are split and returned in a dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to be split into sequences.
    user_indices : Iterable
        Indices to be used what users are to be split into sequences.
    sequence_length : int
        The desired sequence length.

    Returns
    -------
    list[pd.DataFrame]
        Return a list of sequences in form of a dataframe.
    pd.DataFrame
        The rest of user sequences. They are all shorter than the sequence
        length.
    """

    user_sequences = []
    rest_of_user_sequences = []
    for user_id in user_indices:
        user_df = df[df["user_id"] == user_id].sort_values("date")

        while len(user_df) >= sequence_length:
            user_sequences.append(user_df.iloc[:sequence_length])
            user_df = user_df.iloc[sequence_length:]
        rest_of_user_sequences.append(user_df)

    return user_sequences, pd.concat(rest_of_user_sequences)

In [25]:
indexes = merged.value_counts("user_id")[(merged.value_counts("user_id") >= sequence_length)].index

In [26]:
user_sequences, rest_of_user_sequences = split_to_sequences(merged, user_indices=indexes, sequence_length=5)

In [27]:
user_matrix = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in user_sequences]
)

In [28]:
np.save(CUSTOM_DATASET_PATH / "user_matrix_5", user_matrix, allow_pickle=False)

In [29]:
user_matrix = np.load(CUSTOM_DATASET_PATH / "user_matrix_5.npy")

In [30]:
print("user_matrix.shape:", user_matrix.shape)

user_matrix.shape: (32052, 5, 385)


### The rest of more or equal to sequence length

In [31]:
rest_of_user_sequences

,review_id,user_id,business_id,index,latitude,longitude,stars_business,review_count_review,Middle Eastern_category,Argentine_category,...,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos,number_of_friends
22234,AFNqh6PvrZpHrzr_Mh5gpA,2930,299,0.079638,0.284109,0.508034,0.875,0.008397,0.0,0.0,...,0.000318,0.0,0.0,0.011514,0.003669,0.001832,0.001832,0.003369,0.001666,0.009804
182138,xRMMNUcdRRbuzCmwDGFokg,2930,2255,0.668941,0.152727,0.527117,0.375,0.001225,0.0,0.0,...,0.000318,0.0,0.0,0.011514,0.003669,0.001832,0.001832,0.003369,0.001666,0.009804
98239,XKkWOMNPKdAJs99CD3mvYA,293,1193,0.355600,0.622611,0.361358,1.000,0.008572,0.0,0.0,...,0.007952,0.0,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
148321,j8AMTLpOZmkP5IqqUd8zKQ,293,1791,0.544499,0.662821,0.322732,0.625,0.015045,0.0,0.0,...,0.007952,0.0,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
188847,NV_WYM_eRw8Xy_HUUvG1pg,293,2346,0.692102,0.344865,0.539596,0.875,0.006823,0.0,0.0,...,0.007952,0.0,0.0,0.015069,0.010841,0.014611,0.014611,0.011502,0.010211,0.023543
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193236,0KSGC1i-G5dcmNa-IReQnQ,46671,2396,0.707821,0.321163,0.516100,0.750,0.161127,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
260725,BH-Ed8fsfzALscWNT1ySgg,46894,3255,0.956784,0.297953,0.498115,0.750,0.142932,0.0,0.0,...,0.000000,0.0,0.0,0.000169,0.000041,0.000085,0.000085,0.000000,0.000000,0.026344
186959,CMjJRLyM788cLZ6QGU6XbQ,47192,2309,0.684880,0.629683,0.797963,0.750,0.013296,0.0,0.0,...,0.000000,0.0,0.0,0.000169,0.000289,0.000000,0.000000,0.000116,0.000000,0.002268
194925,Hyr0Fh0_4hqm7a7665inJQ,47688,2409,0.713235,0.252180,0.511494,0.625,0.061057,0.0,0.0,...,0.000000,0.0,0.0,0.000169,0.000000,0.000000,0.000000,0.000000,0.000000,0.006002


In [32]:
indexes_less_than_sequence_length = rest_of_user_sequences.value_counts("user_id")[(rest_of_user_sequences.value_counts("user_id") > 1) & (rest_of_user_sequences.value_counts("user_id") < sequence_length)].index

In [33]:
rest_of_user_sequences_padding = split_to_sequences_padding(df=rest_of_user_sequences, user_indices=indexes_less_than_sequence_length, sequence_length=sequence_length)

In [34]:
rest_of_user_matrix_padding = np.array(
    [user_df.drop(columns=["review_id"]).to_numpy() for user_df in rest_of_user_sequences_padding]
)

In [35]:
np.save(CUSTOM_DATASET_PATH / "rest_of_user_matrix_padding_5", rest_of_user_matrix_padding, allow_pickle=False)

In [36]:
rest_of_user_matrix_padding = np.load(CUSTOM_DATASET_PATH / "rest_of_user_matrix_padding_5.npy")

In [37]:
print("rest_of_user_matrix_padding.shape:", rest_of_user_matrix_padding.shape)

rest_of_user_matrix_padding.shape: (6007, 5, 385)


## Merging all the sequences

In [38]:
# merge all the matrices
final_user_matrix = np.concat([user_matrix, user_matrix_padding, rest_of_user_matrix_padding])
print("final_user_matrix.shape:", final_user_matrix.shape)

final_user_matrix.shape: (74235, 5, 385)


In [39]:
user_matrix_x = final_user_matrix[:, :sequence_length-1, :]
print("user_matrix_x.shape:", user_matrix_x.shape)

user_matrix_x.shape: (74235, 4, 385)


In [40]:
user_ids_x = user_matrix_x[:, :, 0]
restaurant_ids_x = user_matrix_x[:, :, 1]
attributes_x = user_matrix_x[:, :, 2:]
print("user_ids_x.shape:", user_ids_x.shape)
print("restaurant_ids_x.shape:", restaurant_ids_x.shape)
print("attributes_x.shape:", attributes_x.shape)

user_ids_x.shape: (74235, 4)
restaurant_ids_x.shape: (74235, 4)
attributes_x.shape: (74235, 4, 383)


In [41]:
user_matrix_y = final_user_matrix[:, sequence_length-1:, 1:2]
print("user_matrix_y.shape:", user_matrix_y.shape)

user_matrix_y.shape: (74235, 1, 1)


## Train/Test split

In [42]:
user_ids_x_train, user_ids_x_test, restaurant_ids_x_train, restaurant_ids_x_test, attributes_x_train, attributes_x_test, y_train, y_test = train_test_split(user_ids_x, restaurant_ids_x, attributes_x, user_matrix_y, train_size=0.8, random_state=28)

In [43]:
print("user_ids_x_train.shape:", user_ids_x_train.shape)
print("user_ids_x_train.dtype:", user_ids_x_train.dtype)
print("user_ids_x_test.shape:", user_ids_x_test.shape)
print("user_ids_x_test.dtype:", user_ids_x_test.dtype)
print("restaurant_ids_x_train.shape:", restaurant_ids_x_train.shape)
print("restaurant_ids_x_train.dtype:", restaurant_ids_x_train.dtype)
print("restaurant_ids_x_test.shape:", restaurant_ids_x_test.shape)
print("restaurant_ids_x_test.dtype:", restaurant_ids_x_test.dtype)
print("attributes_x_train.shape:", attributes_x_train.shape)
print("attributes_x_train.dtype:", attributes_x_train.dtype)
print("attributes_x_test.shape:", attributes_x_test.shape)
print("attributes_x_test.dtype:", attributes_x_test.dtype)
print("y_train.shape:", y_train.shape)
print("y_train.dtype:", y_train.dtype)
print("y_test.shape:", y_test.shape)
print("y_test.dtype:", y_test.dtype)

user_ids_x_train.shape: (59388, 4)
user_ids_x_train.dtype: float64
user_ids_x_test.shape: (14847, 4)
user_ids_x_test.dtype: float64
restaurant_ids_x_train.shape: (59388, 4)
restaurant_ids_x_train.dtype: float64
restaurant_ids_x_test.shape: (14847, 4)
restaurant_ids_x_test.dtype: float64
attributes_x_train.shape: (59388, 4, 383)
attributes_x_train.dtype: float64
attributes_x_test.shape: (14847, 4, 383)
attributes_x_test.dtype: float64
y_train.shape: (59388, 1, 1)
y_train.dtype: float64
y_test.shape: (14847, 1, 1)
y_test.dtype: float64


In [44]:
config = ConfigParser()
config.read("credentials.ini")
wandb_api_key = config["WandB"]["api_key"]

In [45]:
# hyperparameters and stuff
name = "recommendLayerNorm"
version = 12
learning_rate = 0.00001
number_of_epochs = 85
batch_size = 1024
user_embedding_dim = 28
restaurant_embedding_dim = 64
lstm_out_units = 112
logged = False

In [46]:
restaurant_input = Input(shape=(sequence_length-1,), dtype='float64', name='restaurant_ids')
user_input = Input(shape=(sequence_length-1,), dtype='float64', name='user_ids')
attributes_input = Input(shape=(sequence_length-1, attributes_x_train.shape[-1]), dtype='float64', name='attributes')

In [47]:
restaurants_ids = final_user_matrix[:, :, 1]
num_restaurants = int(restaurants_ids.max()) + 1 # zero is padding

In [48]:
user_ids = final_user_matrix[:, :, 0]
num_users = len(np.unique(user_ids))

In [49]:
restaurant_embedding = Embedding(
        input_dim=num_restaurants,
        output_dim=restaurant_embedding_dim,
        mask_zero=True,
        name='restaurant_embedding'
    )(restaurant_input)

user_embedding = Embedding(
        input_dim=num_users,
        output_dim=user_embedding_dim,
        name='user_embedding'
    )(user_input)

combined = Concatenate(axis=-1)([
        restaurant_embedding,
        user_embedding,
        attributes_input,
    ])

lstm_out = LSTM(lstm_out_units, return_sequences=False)(combined)
lstm_out_layer_norm = LayerNormalization()(lstm_out)
output = Dense(num_restaurants, activation='softmax')(lstm_out_layer_norm)

E0000 00:00:1777219605.847232  314117 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [50]:
model = keras.Model(
    inputs=[restaurant_input, user_input, attributes_input],
    outputs=output
)

optimizer = keras.optimizers.Adam()
optimizer.learning_rate.assign(learning_rate)

model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=[
        'accuracy',
        SparseTopKCategoricalAccuracy(k=5, name='top_5_acc'),
        SparseTopKCategoricalAccuracy(k=10, name='top_10_acc')
    ]
)

model_summary = ""
def save_summary(line):
    global model_summary
    model_summary = model_summary + line

model.summary(print_fn=save_summary)

In [51]:
print(model_summary)

Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ restaurant_ids      │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ restaurant_embeddi… │ (None, 4, 64)     │    219,712 │ restaurant_ids[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_ids            │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attributes          │ (None, 4, 383)    │          0 │

In [52]:
class ModelCallback(keras.callbacks.Callback):
    def __init__(self, restaurant_ids_x_test, user_ids_x_test, attributes_x_test, y_test, batch_size, logged=True):
        self.x_test = [restaurant_ids_x_test, user_ids_x_test, attributes_x_test]
        self.y_test = y_test
        self.batch_size = batch_size
        self.logged = logged

    def on_epoch_end(self, epoch, logs=None):
        new_logs = dict()
        for metric in logs:
            new_logs["train_" + metric] = logs[metric]
        evaluation = self.model.evaluate(x=self.x_test, y=self.y_test, batch_size=self.batch_size)
        loss = evaluation[0]
        accuracy = evaluation[1]
        top_5_accuracy = evaluation[2]
        top_10_accuracy = evaluation[3]
        new_logs["test_loss"] = loss
        new_logs["test_accuracy"] = accuracy
        new_logs["test_top_5_accuracy"] = top_5_accuracy
        new_logs["test_top_10_accuracy"] = top_10_accuracy
        if self.logged:
            run.log(new_logs)

model_callback = ModelCallback(
    restaurant_ids_x_test=restaurant_ids_x_test,
    user_ids_x_test=user_ids_x_test,
    attributes_x_test=attributes_x_test,
    y_test=y_test,
    batch_size=batch_size,
    logged=logged
)

In [53]:
if logged:
    print("Logging = On!")
    run = wandb.init(
        entity="ISA2026",
        project="ISA_LSTM_2",
        name=name + "." + str(version),
        config={
            "learning_rate": learning_rate,
            "number_of_epochs": number_of_epochs,
            "batch_size": batch_size,
            "user_embedding_dim": user_embedding_dim,
            "restaurant_embedding_dim": restaurant_embedding_dim,
            "model_summary": model_summary,
            "lstm_out_units": lstm_out_units,
        },
    )
    run.log_code(".", include_fn=lambda path: path.endswith(".ipynb"))
else:
    print("Logging = Off!")

Logging = Off!


In [54]:
history = model.fit(
    [restaurant_ids_x_train, user_ids_x_train, attributes_x_train],
    y_train,
    batch_size=batch_size,
    epochs=number_of_epochs,
    verbose=2,
    callbacks=[model_callback]
)
if logged:
    run.finish()

Epoch 1/85


E0000 00:00:1777219608.836909  314117 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 2.9843e-04 - loss: 8.1362 - top_10_acc: 0.0040 - top_5_acc: 0.0023
58/58 - 5s - 85ms/step - accuracy: 1.4169e-04 - loss: 8.1558 - top_10_acc: 0.0026 - top_5_acc: 0.0013
Epoch 2/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0067 - loss: 8.0950 - top_10_acc: 0.0313 - top_5_acc: 0.0259
58/58 - 3s - 52ms/step - accuracy: 7.3085e-04 - loss: 8.1127 - top_10_acc: 0.0195 - top_5_acc: 0.0144
Epoch 3/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0171 - loss: 8.0532 - top_10_acc: 0.0371 - top_5_acc: 0.0321
58/58 - 3s - 55ms/step - accuracy: 0.0151 - loss: 8.0700 - top_10_acc: 0.0346 - top_5_acc: 0.0293
Epoch 4/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0179 - loss: 8.0106 - top_10_acc: 0.0422 - top_5_acc: 0.0339
58/58 - 3s - 57ms/step - accuracy: 0.0173 - loss: 8.0266 - top_10_acc: 0.0399 - top_5_acc: 0.0331
Epoch 5/85
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0183 - loss: 7.9672 - top_10_acc: 0.04

## Naive Bayes